# RecoFix vs. original NuGraphAnalyzer -- comparison plots

Reads a `hist_nugraph_analyser_recofix*.root` file (produced by running both
`ICARUSNuGraphAnalyzer` and `ICARUSNuGraphAnalyzerRecoFix` in the same fcl)
and reproduces the two comparison figures:

1. `icluster`/`ipfp` -- original (uninitialized) vs RecoFix (real association)
2. `NuGraphEventOutput` entry counts (dead-tree fix) + NG score sanity check

Requires: `uproot`, `pandas`, `matplotlib`
```
pip install uproot pandas matplotlib --break-system-packages
```

In [ ]:
import uproot
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

ORIG_DIR = "NuGraphAnaCryoE"
FIX_DIR = "NuGraphAnaCryoERecoFix"

## Load the file

Edit `root_path` to point at your own output file.

In [ ]:
root_path = "hist_nugraph_analyser_recofix_trainingsample.root"

f = uproot.open(root_path)
orig_hits = f[f"{ORIG_DIR}/NuGraphHitOutput"].arrays(library="pd")
fix_hits = f[f"{FIX_DIR}/NuGraphHitOutput"].arrays(library="pd")
orig_evt_entries = f[f"{ORIG_DIR}/NuGraphEventOutput"].num_entries
fix_evt_entries = f[f"{FIX_DIR}/NuGraphEventOutput"].num_entries

print(f"row counts -- original: {len(orig_hits)}  recofix: {len(fix_hits)}")
print(f"NuGraphEventOutput entries -- original: {orig_evt_entries}  recofix: {fix_evt_entries}")

## Plot 1 -- `icluster` / `ipfp`

Original: both columns are class members that were never assigned in
`analyze()` -- every hit should read exactly `0`, not a real association.
RecoFix: real values from an actual `PFParticle->Cluster->Hit` walk, `-1`
where a hit genuinely has no match.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(orig_hits["icluster"], bins=50, color="#C0392B", alpha=0.8)
axes[0, 0].set_title("icluster -- ORIGINAL (uninitialized)")
axes[0, 0].set_xlabel("icluster value")
axes[0, 0].set_ylabel("hit count")

axes[0, 1].hist(fix_hits["icluster"], bins=50, color="#1B8A5A", alpha=0.8)
axes[0, 1].set_title("icluster -- RECOFIX (real association)")
axes[0, 1].set_xlabel("icluster value")
axes[0, 1].set_ylabel("hit count")

axes[1, 0].hist(orig_hits["ipfp"], bins=30, color="#C0392B", alpha=0.8)
axes[1, 0].set_title("ipfp -- ORIGINAL (uninitialized)")
axes[1, 0].set_xlabel("ipfp value")
axes[1, 0].set_ylabel("hit count")

axes[1, 1].hist(fix_hits["ipfp"], bins=30, color="#1B8A5A", alpha=0.8)
axes[1, 1].set_title("ipfp -- RECOFIX (real association)")
axes[1, 1].set_xlabel("ipfp value")
axes[1, 1].set_ylabel("hit count")

plt.tight_layout()
plt.savefig("icluster_ipfp_comparison.png", dpi=130)
plt.show()

## Plot 2 -- `NuGraphEventOutput` entries + NG score sanity check

Left: entry count in the event-level tree (should be `0` for the original,
one row per processed event for RecoFix). Right: a chosen NG score
(`shower` by default) plotted original-vs-RecoFix per hit -- RecoFix only
touches bookkeeping columns, so every point should sit exactly on `y=x`.

In [ ]:
score_branch = "shower"  # try "MIP", "HIP", "michel", "diffuse" too

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = [orig_evt_entries, fix_evt_entries]
axes[0].bar(["ORIGINAL", "RECOFIX"], counts, color=["#C0392B", "#1B8A5A"])
axes[0].set_title("NuGraphEventOutput entries")
axes[0].set_ylabel("rows in tree")
for i, v in enumerate(counts):
    axes[0].text(i, v + 0.05, str(v), ha="center", fontweight="bold")

axes[1].scatter(
    orig_hits[score_branch], fix_hits[score_branch],
    s=2, alpha=0.3, color="#1E2761",
)
axes[1].plot([0, 1], [0, 1], color="#C0392B", linewidth=1, linestyle="--")
axes[1].set_xlabel(f"{score_branch} score -- ORIGINAL")
axes[1].set_ylabel(f"{score_branch} score -- RECOFIX")
axes[1].set_title(f"NG '{score_branch}' score unchanged (should sit exactly on y=x)")

plt.tight_layout()
plt.savefig("eventoutput_and_scorecheck.png", dpi=130)
plt.show()